## Configurações, Imports e Funções

### Imports

In [51]:
import os
import re
import json
import math
import warnings
from glob import glob
from collections import defaultdict, Counter
import ast

import pandas as pd
import numpy as np
from pandarallel import pandarallel

# Gensim
from gensim import corpora, models
from gensim.models import LdaModel
from gensim.corpora import Dictionary

# Scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# XGBoost
from xgboost import XGBClassifier

# Imbalanced-learn
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

### Configurações

In [52]:
# Configurações
warnings.filterwarnings("ignore", category=DeprecationWarning)
pandarallel.initialize(progress_bar=True)

# Constantes de configuração
RANDOM_STATE = 42
NUMERO_DE_TOPICOS = 5
WORDS_PER_TOPIC_FOR_DISPLAY = 8
NUM_PALAVRAS_POR_TOPICO = 5
PERCENTUAL_SIMILARIDADE = 0.95

# Caminhos dos arquivos
DATASET_PATH = './datasets/database-lemmetizado.csv.zip'
FILTERED_IPCR_PATH = './datasets/database-filtrado.csv.zip'
MODELS_DIR = f'./modelos/lda_por_ano/{NUMERO_DE_TOPICOS}_topics/'
RESULTS_DIR = f'./resultados_por_ano/{NUMERO_DE_TOPICOS}_topics/'
ANALYSIS_DIR = f'./resultados_analise/{NUMERO_DE_TOPICOS}_topics/'


INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.

https://nalepae.github.io/pandarallel/troubleshooting/


### Funções

#### Funções Auxiliares

In [53]:
def load_dataset(filepath, compression='zip'):
    """Carrega o dataset e prepara para processamento."""
    df = pd.read_csv(filepath, compression=compression)
    df = df.sort_values('date_published').reset_index(drop=True)
    df['date_published'] = pd.to_datetime(df['date_published'], errors='coerce')
    return df


def tokenize_text(text):
    """Tokeniza o texto convertendo para minúsculas e separando por espaços."""
    return text.lower().split()


def prepare_corpus_and_dictionary(df, token_column='tokens'):
    """Prepara corpus e dicionário para modelagem LDA."""
    documents = df[token_column].dropna().astype(str).tolist()
    processed_docs = [tokenize_text(doc) for doc in documents]
    
    dictionary = corpora.Dictionary(processed_docs)
    corpus = [dictionary.doc2bow(doc) for doc in processed_docs]
    
    return corpus, dictionary, processed_docs

#### Funções de Treinamento do LDA

In [54]:
def train_and_save_lda_per_year(df, num_topics=10, passes=10, 
                                alpha='auto', eta='auto', output_dir=MODELS_DIR):
    """
    Treina um modelo LDA para cada ano e salva-o individualmente.
    
    Args:
        df: DataFrame com 'date_published' e 'tokens'
        num_topics: Número de tópicos para cada modelo
        passes: Número de passes de treinamento
        alpha: Parâmetro alpha do LDA
        eta: Parâmetro eta do LDA
        output_dir: Diretório para salvar os modelos
    """
    df['year'] = df['date_published'].dt.year
    years = sorted(df['year'].dropna().unique())
    
    print(f"Anos encontrados no dataset: {years}")
    
    os.makedirs(output_dir, exist_ok=True)
    print(f"Modelos serão salvos em: '{output_dir}'")
    
    models_trained_count = 0
    
    for year in years:
        print(f"\n=== Processando ano: {year} ===")
        
        df_year = df[df['year'] == year].copy()
        documents = df_year['tokens'].dropna().tolist()
        
        processed_docs = [
            doc.split() if isinstance(doc, str) else doc 
            for doc in documents
        ]
        processed_docs = [doc for doc in processed_docs if doc]
        
        print(f"Documentos para treinamento: {len(processed_docs)}")
        
        dictionary_year = corpora.Dictionary(processed_docs)
        corpus_year = [dictionary_year.doc2bow(doc) for doc in processed_docs]
        
        try:
            lda_model = models.LdaModel(
                corpus=corpus_year,
                id2word=dictionary_year,
                num_topics=num_topics,
                random_state=RANDOM_STATE,
                passes=passes,
                alpha=alpha,
                eta=eta
            )
            
            model_path = os.path.join(output_dir, f'lda_model_{year}.model')
            lda_model.save(model_path)
            
            print(f"Modelo salvo: '{model_path}'")
            models_trained_count += 1
            
        except Exception as e:
            print(f"ERRO ao treinar o modelo para o ano {year}: {e}")
            continue
    
    print(f"\n=== {models_trained_count} modelos treinados com sucesso ===")


def display_topics_from_models_with_probs(model_dir, num_words=10):
    """Exibe os tópicos de cada modelo LDA com probabilidades."""
    if not os.path.exists(model_dir):
        print(f"ERRO: O diretório '{model_dir}' não foi encontrado.")
        return

    model_files = [
        f for f in os.listdir(model_dir) 
        if f.startswith('lda_model_') and f.endswith('.model')
    ]
    
    if not model_files:
        print(f"Nenhum modelo encontrado em '{model_dir}'.")
        return
    
    model_files.sort()
    print(f"Encontrados {len(model_files)} modelos.\n")
    
    for filename in model_files:
        try:
            match = re.search(r'_(\d{4})\.model', filename)
            if not match:
                continue
            
            year = match.group(1)
            model_path = os.path.join(model_dir, filename)
            lda_model = LdaModel.load(model_path)
            
            print(f"--- Tópicos para o Ano: {year} ---")
            
            topics = lda_model.show_topics(
                num_topics=-1, 
                num_words=num_words, 
                formatted=False
            )
            
            for topic_id, word_probs in topics:
                formatted_words = [
                    f"{word} ({prob*100:.2f}%)" 
                    for word, prob in word_probs
                ]
                print(f"Tópico {topic_id}: {', '.join(formatted_words)}")
            print()
                
        except Exception as e:
            print(f"ERRO ao processar {filename}: {e}")

#### Funções de processamento de patentes

In [55]:
def _processar_patente(row, lda_model, dicionario, 
                      distribuicoes_topicos_ano, num_topics):
    """Processa uma única patente extraindo distribuição de tópicos."""
    lens_id_final = row['lens_id']
    tokens_brutos = row['tokens']

    # filtros de tokens inválidos
    if pd.isna(tokens_brutos) or tokens_brutos == '':
        return None
    
    if isinstance(tokens_brutos, str):
        tokens_patente = tokens_brutos.split()
    elif isinstance(tokens_brutos, list):
        tokens_patente = tokens_brutos
    else:
        return None
    
    tokens_patente = [t for t in tokens_patente if t and isinstance(t, str)]
    
    if not tokens_patente:
        return None

    # distribuição de tópicos para a patente
    doc_bow = dicionario.doc2bow(tokens_patente)
    distribuicao_topicos_patente = lda_model.get_document_topics(
        doc_bow, minimum_probability=0.0
    )
    prob_topico_dado_doc = {
        topico_id: prob 
        for topico_id, prob in distribuicao_topicos_patente
    }

    resultado_patente = {
        'lens_id': lens_id_final, 
        'year': int(row['year'])
    }
    
    # calcula a pontuação das palavras por tópico
    for id_topico in range(num_topics):
        p_topico_na_patente = prob_topico_dado_doc.get(id_topico, 0)
        distribuicao_palavras_topico = distribuicoes_topicos_ano[id_topico]
        
        lista_pontuacao_palavras = []
        doc_bow_dict = dict(doc_bow)
        
        for id_palavra, p_palavra_no_topico in distribuicao_palavras_topico:
            if id_palavra in doc_bow_dict:
                pontuacao = float(p_topico_na_patente * p_palavra_no_topico)
                palavra_str = dicionario[id_palavra]
                lista_pontuacao_palavras.append((palavra_str, pontuacao))
        
        lista_pontuacao_palavras.sort(key=lambda item: item[1], reverse=True)
        top_palavras = lista_pontuacao_palavras[:50]
        
        resultado_patente[f'Topic_{id_topico}'] = json.dumps(
            top_palavras, ensure_ascii=False
        )
    
    return resultado_patente


def analisar_e_salvar_por_ano_aprimorado(df, modelos_dir, num_topics, 
                                         output_dir, anos_para_rodar=None, 
                                         ignorar_existentes=True):
    """Processa patentes ano por ano de forma otimizada."""
    os.makedirs(output_dir, exist_ok=True)
    
    colunas_necessarias = ['lens_id', 'year', 'tokens']
    for col in colunas_necessarias:
        if col not in df.columns:
            raise ValueError(f"Coluna '{col}' não encontrada!")
    
    if df['lens_id'].duplicated().any():
        duplicados = df['lens_id'].duplicated().sum()
        print(f"AVISO: {duplicados} lens_id duplicados. Removendo...")
        df = df.drop_duplicates(subset=['lens_id'], keep='first')
    
    if anos_para_rodar:
        anos_alvo = (
            [anos_para_rodar] if isinstance(anos_para_rodar, int) 
            else sorted(anos_para_rodar)
        )
    else:
        anos_alvo = sorted(df['year'].dropna().unique().astype(int))
    
    print(f"Iniciando análise para os anos: {anos_alvo}")

    for ano in anos_alvo:
        if ignorar_existentes:
            caminho_parquet = os.path.join(output_dir, f'resultados_{ano}.parquet')
            caminho_csv = os.path.join(output_dir, f'resultados_{ano}.csv')
            
            if os.path.exists(caminho_parquet) or os.path.exists(caminho_csv):
                print(f"\n--- Ano {ano} já processado. Pulando... ---")
                continue

        print(f"\n{'='*60}")
        print(f"Processando o ano: {ano}")
        print(f"{'='*60}")
        
        caminho_modelo = os.path.join(modelos_dir, f'lda_model_{ano}.model')
        
        if not os.path.exists(caminho_modelo):
            print(f"AVISO: Modelo para {ano} não encontrado")
            continue
            
        try:
            lda_model_ano = LdaModel.load(caminho_modelo)
            dicionario_ano = lda_model_ano.id2word
            print("Modelo carregado")
        except Exception as e:
            print(f"ERRO ao carregar modelo: {e}")
            continue

        print("Pré-calculando distribuições de tópicos...")
        distribuicoes_topicos_ano = {
            id_topico: lda_model_ano.get_topic_terms(id_topico, topn=500)
            for id_topico in range(num_topics)
        }
        
        df_ano = df[df['year'] == ano].copy()
        print(f"{len(df_ano)} patentes para processar")

        if df_ano.empty:
            print("Nenhuma patente para processar.")
            continue
        
        print("Processando patentes...")
        try:
            resultados_do_ano_series = df_ano.parallel_apply(
                _processar_patente, 
                axis=1, 
                lda_model=lda_model_ano, 
                dicionario=dicionario_ano,
                distribuicoes_topicos_ano=distribuicoes_topicos_ano, 
                num_topics=num_topics
            )
        except Exception as e:
            print(f"ERRO: {e}")
            import traceback
            traceback.print_exc()
            continue
        
        resultados_do_ano_lista = [
            res for res in resultados_do_ano_series if res is not None
        ]
        
        if not resultados_do_ano_lista:
            print("Nenhum resultado válido.")
            continue
        
        print(f"{len(resultados_do_ano_lista)} patentes processadas")
            
        df_resultados_ano = pd.DataFrame(resultados_do_ano_lista)
        
        caminho_saida_parquet = os.path.join(output_dir, f'resultados_{ano}.parquet')
        caminho_saida_csv = os.path.join(output_dir, f'resultados_{ano}.csv')
        
        try:
            df_resultados_ano.to_parquet(
                caminho_saida_parquet, index=False, engine='pyarrow'
            )
            print(f"✓ Salvo em Parquet: {caminho_saida_parquet}")
        except Exception as e:
            print(f"⚠ Erro ao salvar Parquet, usando CSV: {str(e)[:100]}")
            try:
                df_resultados_ano.to_csv(caminho_saida_csv, index=False)
                print(f"✓ Salvo em CSV: {caminho_saida_csv}")
            except Exception as e2:
                print(f"✗ ERRO ao salvar: {e2}")

    print("\n" + "="*60)
    print("PROCESSAMENTO CONCLUÍDO")
    print("="*60)


#### Funções de Tratamento de Resultados

In [56]:
def concatenar_resultados_por_ano(diretorio_resultados, formato='csv'):
    """Concatena todos os arquivos de resultados em um único DataFrame."""
    if not os.path.exists(diretorio_resultados):
        raise ValueError(f"Diretório '{diretorio_resultados}' não encontrado!")
    
    padroes_formato = {
        'auto': ['resultados_*.csv', 'resultados_*.parquet'],
        'csv': ['resultados_*.csv'],
        'parquet': ['resultados_*.parquet']
    }
    
    padroes = padroes_formato.get(formato)
    if not padroes:
        raise ValueError("formato deve ser 'csv', 'parquet' ou 'auto'")
    
    arquivos = []
    for padrao in padroes:
        caminho_completo = os.path.join(diretorio_resultados, padrao)
        arquivos.extend(glob(caminho_completo))
    
    if not arquivos:
        print(f"⚠ Nenhum arquivo encontrado em '{diretorio_resultados}'")
        return pd.DataFrame()
    
    arquivos = sorted(set(arquivos))
    print(f"Encontrados {len(arquivos)} arquivos para concatenar")
    print(f"{'='*60}")
    
    dataframes_lista = []
    
    for arquivo in arquivos:
        try:
            nome_arquivo = os.path.basename(arquivo)
            
            if arquivo.endswith('.csv'):
                df_temp = pd.read_csv(arquivo)
                tipo = 'CSV'
            elif arquivo.endswith('.parquet'):
                df_temp = pd.read_parquet(arquivo)
                tipo = 'Parquet'
            else:
                continue
            
            dataframes_lista.append(df_temp)
            print(f"✓ {nome_arquivo} ({tipo}) - {len(df_temp)} registros")
            
        except Exception as e:
            print(f"✗ ERRO ao carregar {nome_arquivo}: {str(e)[:80]}")
            continue
    
    if not dataframes_lista:
        print("\n⚠ Nenhum DataFrame válido foi carregado!")
        return pd.DataFrame()
    
    print(f"\n{'='*60}")
    print(f"Concatenando {len(dataframes_lista)} DataFrames...")
    
    df_final = pd.concat(dataframes_lista, ignore_index=True)
    
    print("Concatenação concluída!")
    print(f"\nRESUMO DO DATASET FINAL:")
    print(f"{'='*60}")
    print(f" --- Total de registros: {len(df_final):,}")
    print(f" --- Total de colunas: {len(df_final.columns)}")
    print(f" --- Anos presentes: {sorted(df_final['year'].unique().tolist())}")
    print(f" --- Registros por ano:")
    
    contagem_por_ano = df_final['year'].value_counts().sort_index()
    for ano, contagem in contagem_por_ano.items():
        print(f"    - {ano}: {contagem:,} patentes")
    
    print(f"{'='*60}")
    
    return df_final


def converter_json_para_listas(df, prefixo_coluna='Topic_'):
    """Converte colunas JSON (strings) de volta para listas Python."""
    df_copia = df.copy()
    colunas_topicos = [
        col for col in df_copia.columns if col.startswith(prefixo_coluna)
    ]
    
    print(f"Convertendo {len(colunas_topicos)} colunas de tópicos...")
    
    for col in colunas_topicos:
        try:
            df_copia[col] = df_copia[col].apply(
                lambda x: json.loads(x) 
                if isinstance(x, str) and x.strip() != '[]' 
                else []
            )
        except Exception as e:
            print(f"⚠ Erro ao converter coluna {col}: {e}")
            continue
    
    print("Conversão concluída!")
    return df_copia


#### Funções de Análise de Similaridade

In [57]:
def clean_word_original(palavra_suja):
    """Limpa palavras removendo caracteres não-alfabéticos."""
    if isinstance(palavra_suja, str):
        palavra_limpa = re.sub(r"[^a-zA-Zá-úÁ-Ú]", "", palavra_suja)
        return palavra_limpa if palavra_limpa else None
    return None

# Analisa de forma a fazer a pergunta:
# Estas patentes têm as mesmas palavras (sem considerar seus pesos)?
def analise_similaridade_jaccard(df_completo, numero_palavras_por_topico, 
                                 percentual_similaridade, numero_topicos):
    """Análise de similaridade usando coeficiente de Jaccard otimizado."""
    df_otimizado = df_completo.copy()
    topic_columns = [col for col in df_otimizado.columns if 'Topic_' in col]
    
    print("Iniciando extração vetorizada de palavras-chave...")

    # 'Melt' transforma colunas (Topic_1, Topic_2) em linhas
    df_melted = df_otimizado.melt(
        id_vars=['lens_id'], 
        value_vars=topic_columns, 
        value_name='topic_list'
    )
    
    df_melted = df_melted.dropna(subset=['topic_list'])

    # 'Explode' transforma listas em linhas
    # Se 'topic_list' era [['word1', 0.5], ['word2', 0.4]],
    # agora teremos duas linhas: ['word1', 0.5] e ['word2', 0.4]
    df_exploded = df_melted.explode('topic_list')
    
    df_exploded = df_exploded[
        df_exploded['topic_list'].apply(
            lambda x: isinstance(x, list) and len(x) > 0
        )
    ]
    
    df_exploded['palavra_suja'] = df_exploded['topic_list'].str[0]
    
    print("Limpando palavras-chave...")
    df_exploded['palavra_limpa'] = df_exploded['palavra_suja'].apply(
        clean_word_original
    )
    
    df_exploded = df_exploded.dropna(subset=['palavra_limpa'])

    # Agrupa por patente E por tópico de origem, e pega as N primeiras
    # Isso simula o '[:n_palavras]' da função original, para CADA tópico
    df_top_n = df_exploded.groupby(['lens_id', 'variable']).head(
        numero_palavras_por_topico
    )
    
    palavras_chave_series = df_top_n.groupby('lens_id')['palavra_limpa'].apply(set)
    df_otimizado['palavras_chave'] = df_otimizado['lens_id'].map(
        palavras_chave_series
    )
    df_otimizado['palavras_chave'] = df_otimizado['palavras_chave'].apply(
        lambda x: x if isinstance(x, set) else set()
    )
    
    print("Assinaturas de palavras criadas com sucesso.")
    
    print("\nIniciando comparação de similaridade otimizada...")
    
    resultados_finais = []
    anos_unicos = sorted(df_otimizado['year'].unique())
    
    for i in range(len(anos_unicos) - 1):
        ano_atual = anos_unicos[i]
        ano_seguinte = anos_unicos[i+1]
        
        print(f"Comparando ano {ano_atual} com {ano_seguinte}...")
        
        df_ano_atual = df_otimizado[df_otimizado['year'] == ano_atual]
        df_ano_seguinte = df_otimizado[df_otimizado['year'] == ano_seguinte]
        
        # Cria um mapa de lookup rápido para o ano seguinte: {lens_id -> set_de_palavras}
        palavras_seguintes_map = df_ano_seguinte.set_index('lens_id')['palavras_chave']
        
        # Cria o Índice Invertido para o ano seguinte
        # Formato: {palavra -> {id1, id2, ...}}
        inverted_index = defaultdict(set)
        for lens_id, palavras_set in palavras_seguintes_map.items():
            for palavra in palavras_set:
                inverted_index[palavra].add(lens_id)

        # Itera sobre o ano atual        
        for patente_atual in df_ano_atual.itertuples():
            id_atual = patente_atual.lens_id
            palavras_atuais = patente_atual.palavras_chave
            len_atuais = len(palavras_atuais)
            
            if len_atuais == 0:
                resultados_finais.append({
                    'lens_id': id_atual,
                    'patentes_similares': []
                })
                continue
            
            # Encontra candidatos usando o Índice Invertido
            # 'candidate_counts' irá armazenar: {id_candidato -> contagem_de_palavras_em_comum}
            # A contagem de palavras em comum é exatamente o tamanho da INTERSEÇÃO
            candidate_counts = Counter()
            for palavra in palavras_atuais:
                candidate_counts.update(inverted_index[palavra])
                
            patentes_similares_encontradas = []
            
            for id_seguinte, intersecao in candidate_counts.items():
                palavras_seguintes = palavras_seguintes_map[id_seguinte]
                uniao = len_atuais + len(palavras_seguintes) - intersecao
                
                similaridade = intersecao / uniao if uniao > 0 else 0
                
                if similaridade >= percentual_similaridade:
                    patentes_similares_encontradas.append(id_seguinte)
            
            resultados_finais.append({
                'lens_id': id_atual,
                'patentes_similares': patentes_similares_encontradas
            })
    
    print("Comparação finalizada.")
    
    df_resultados = pd.DataFrame(resultados_finais)
    df_otimizado = pd.merge(df_otimizado, df_resultados, on='lens_id', how='left')
    
    df_otimizado['patentes_similares'] = df_otimizado['patentes_similares'].apply(
        lambda x: x if isinstance(x, list) else []
    )
    df_otimizado['count_similares'] = df_otimizado['patentes_similares'].apply(len)
    df_otimizado['emergente'] = df_otimizado['count_similares'] > 0
    
    print("\nResultado Final:")
    print(df_otimizado[['lens_id', 'year', 'patentes_similares', 
                        'count_similares', 'emergente']].head())
    
    os.makedirs(f'{ANALYSIS_DIR}', exist_ok=True)
    output_path = (
        f'{ANALYSIS_DIR}analise-de-similaridade_jaccard_'
        f'n{numero_palavras_por_topico}_s{percentual_similaridade:.2f}.csv'
    )
    df_otimizado.to_csv(output_path, index=True)
    
    print(f"\nArquivo salvo: {output_path}")
    
    return df_otimizado


def calcular_similaridade_tanimoto(vetor1, vetor2):
    """Calcula similaridade de Tanimoto entre dois vetores de características."""
    palavras_comuns = set(vetor1.keys()).intersection(set(vetor2.keys()))
    produto_escalar = sum(vetor1[palavra] * vetor2[palavra] for palavra in palavras_comuns)
    
    if produto_escalar == 0.0:
        return 0.0
    
    mag_quad_vetor1 = sum(prob**2 for prob in vetor1.values())
    mag_quad_vetor2 = sum(prob**2 for prob in vetor2.values())
    
    denominador = mag_quad_vetor1 + mag_quad_vetor2 - produto_escalar
    
    if denominador == 0:
        return 0.0
    
    return produto_escalar / denominador

# Analisa de forma a fazer a pergunta:
# Estas patentes têm as mesmas palavras e probabilidades(considerando seus pesos)?
# ou falam sobre as mesmas palavras com um nível de importância similar?
# Cosseno: encontrar patentes que falam sobre as mesmas coisas, 
# sem se importar com a força da presença das palavras, e sim dos temas.
# Tanimoto: encontrar patentes que falam sobre as mesmas coisas,
# e que são intrinsecamente muito parecidas em seus focos.
def analise_similaridade_tanimoto_or_cosseno(df_completo, numero_palavras_por_topico, 
                                  percentual_similaridade, numero_topicos, 
                                  metodo='tanimoto'):
    """Análise de similaridade usando Tanimoto/Cossenos otimizado."""
    df2 = df_completo.copy()
    topic_columns = [col for col in df2.columns if 'Topic_' in col]
    
    print("Iniciando extração vetorizada de vetores de características...")
    
    # 'Melt' transforma colunas (Topic_1, Topic_2) em linhas
    df_melted = df2.melt(
        id_vars=['lens_id'], 
        value_vars=topic_columns, 
        var_name='topic_source',
        value_name='topic_list'
    )
    
    df_melted = df_melted.dropna(subset=['topic_list'])

    # 'Explode' transforma listas em linhas
    df_exploded = df_melted.explode('topic_list')
    df_exploded = df_exploded.dropna(subset=['topic_list'])
    
    df_exploded['is_valid'] = df_exploded['topic_list'].apply(
        lambda x: isinstance(x, list) and len(x) == 2
    )
    df_exploded = df_exploded[df_exploded['is_valid']]
    
    df_exploded['palavra_suja'] = df_exploded['topic_list'].str[0].astype(str)
    df_exploded['probabilidade'] = pd.to_numeric(
        df_exploded['topic_list'].str[1], errors='coerce'
    )
    
    df_exploded = df_exploded.dropna(subset=['probabilidade'])
    
    df_exploded['palavra_limpa'] = df_exploded['palavra_suja'].str.replace(
        r"[^a-zA-Zá-úÁ-Ú]", "", regex=True
    )
    
    df_exploded = df_exploded[df_exploded['palavra_limpa'] != '']
    
    df_top_n = df_exploded.groupby(['lens_id', 'topic_source']).head(
        numero_palavras_por_topico
    )
    
    df_final_palavras = df_top_n.drop_duplicates(
        subset=['lens_id', 'palavra_limpa'], keep='first'
    )
    
    vetores_series = df_final_palavras.groupby('lens_id').apply(
        lambda x: dict(zip(x['palavra_limpa'], x['probabilidade']))
    )
    
    df2['vetor_caracteristicas'] = df2['lens_id'].map(vetores_series)
    df2['vetor_caracteristicas'] = df2['vetor_caracteristicas'].apply(
        lambda x: x if isinstance(x, dict) else {}
    )
    
    print("Vetores criados com sucesso.")
    
    del df_melted, df_exploded, df_top_n, df_final_palavras, vetores_series
    
    print(f"Pré-calculando magnitudes para o método '{metodo}'...")
    
    if metodo == 'cossenos':
        df2['magnitude'] = df2['vetor_caracteristicas'].apply(
            lambda v: math.sqrt(sum(prob**2 for prob in v.values()))
        )
    elif metodo == 'tanimoto':
        df2['mag_quadrada'] = df2['vetor_caracteristicas'].apply(
            lambda v: sum(prob**2 for prob in v.values())
        )
    else:
        raise ValueError("Método desconhecido para similaridade.")
    
    print("Magnitudes calculadas.")
    
    print(f"\nIniciando comparação de similaridade ({metodo}) otimizada...")
    
    resultados_finais = []
    anos_unicos = sorted(df2['year'].unique())
    
    for i in range(len(anos_unicos) - 1):
        ano_atual = anos_unicos[i]
        ano_seguinte = anos_unicos[i+1]
        
        print(f"Comparando ano {ano_atual} com {ano_seguinte}...")
        
        df_ano_atual = df2[df2['year'] == ano_atual]
        df_ano_seguinte = df2[df2['year'] == ano_seguinte]
        
        # Cria mapas de lookup rápidos para o ano seguinte
        vetores_seguintes_map = df_ano_seguinte.set_index('lens_id')['vetor_caracteristicas']
        
        mag_seguintes_map = df_ano_seguinte.set_index('lens_id')[
            'magnitude' if metodo == 'cossenos' else 'mag_quadrada'
        ]
        
        # Cria o Índice Invertido Ponderado para o ano seguinte
        # Formato: {palavra -> {id1: prob1, id2: prob2, ...}}
        inverted_index = defaultdict(dict)
        for lens_id, vetor in vetores_seguintes_map.items():
            for palavra, prob in vetor.items():
                inverted_index[palavra][lens_id] = prob

        # Itera sobre o ano atual        
        for patente_atual in df_ano_atual.itertuples():
            id_atual = patente_atual.lens_id
            vetor_atual = patente_atual.vetor_caracteristicas
            
            if not vetor_atual:
                resultados_finais.append({
                    'lens_id': id_atual, 
                    'patentes_similares': []
                })
                continue

            # Calcula os produtos escalares para TODOS os candidatos de uma vez
            # 'dot_products' armazenará: {id_candidato -> produto_escalar}
            dot_products = Counter()
            for palavra, prob_atual in vetor_atual.items():
                if palavra in inverted_index:
                    for id_seguinte, prob_seguinte in inverted_index[palavra].items():
                        dot_products[id_seguinte] += prob_atual * prob_seguinte
                
            patentes_similares_encontradas = []
            
            if metodo == 'cossenos':
                mag_atual = patente_atual.magnitude
                if mag_atual == 0:
                    continue
                
                for id_seguinte, produto_escalar in dot_products.items():
                    mag_seguinte = mag_seguintes_map[id_seguinte]
                    if mag_seguinte == 0:
                        continue
                    
                    similaridade = produto_escalar / (mag_atual * mag_seguinte)
                    if similaridade >= percentual_similaridade:
                        patentes_similares_encontradas.append(id_seguinte)
            
            elif metodo == 'tanimoto':
                mag_sq_atual = patente_atual.mag_quadrada
                
                for id_seguinte, produto_escalar in dot_products.items():
                    mag_sq_seguinte = mag_seguintes_map[id_seguinte]
                    denominador = mag_sq_atual + mag_sq_seguinte - produto_escalar
                    
                    if denominador == 0:
                        continue
                    
                    similaridade = produto_escalar / denominador
                    if similaridade >= percentual_similaridade:
                        patentes_similares_encontradas.append(id_seguinte)
            
            resultados_finais.append({
                'lens_id': id_atual,
                'patentes_similares': patentes_similares_encontradas
            })
    
    print("Comparação finalizada.")
    
    df_resultados = pd.DataFrame(resultados_finais)
    df2 = pd.merge(df2, df_resultados, on='lens_id', how='left')
    
    df2['patentes_similares'] = df2['patentes_similares'].apply(
        lambda x: x if isinstance(x, list) else []
    )
    df2['count_similares'] = df2['patentes_similares'].apply(len)
    df2['emergente'] = df2['count_similares'] > 0
    
    if 'magnitude' in df2.columns:
        df2 = df2.drop(columns=['magnitude'])
    if 'mag_quadrada' in df2.columns:
        df2 = df2.drop(columns=['mag_quadrada'])
    
    print("\nResultado Final:")
    print(df2[['lens_id', 'year', 'patentes_similares', 
               'count_similares', 'emergente']].head())
    
    output_path = (
        f'{ANALYSIS_DIR}analise-de-similaridade_{metodo}_'
        f'n{numero_palavras_por_topico}_s{percentual_similaridade:.2f}.csv'
    )
    df2.to_csv(output_path, index=True)
    
    print(f"\n✓ Arquivo salvo: {output_path}")
    
    return df2


#### Funções para Carregar resultados

In [58]:
def parse_set_safe(val):
    """Converte string representando set para tipo set Python."""
    if pd.isna(val):
        return set()
    try:
        result = ast.literal_eval(val)
        return set(result) if isinstance(result, set) else set(result)
    except (ValueError, SyntaxError, TypeError):
        return set()


def parse_list_safe(val):
    """Converte string representando lista para tipo list Python."""
    if pd.isna(val):
        return []
    try:
        result = ast.literal_eval(val)
        return result if isinstance(result, list) else list(result)
    except (ValueError, SyntaxError, TypeError):
        return []


def parse_dict_safe(val):
    """Converte string representando dicionário para tipo dict Python."""
    if pd.isna(val):
        return {}
    try:
        result = ast.literal_eval(val)
        return result if isinstance(result, dict) else {}
    except (ValueError, SyntaxError, TypeError):
        return {}


def carregar_analise_jaccard(numero_topicos, numero_palavras_por_topico, 
                             percentual_similaridade):
    """Carrega resultados da análise de similaridade Jaccard."""
    caminho = (
        f'{ANALYSIS_DIR}analise-de-similaridade_jaccard_'
        f'n{numero_palavras_por_topico}_s{percentual_similaridade:.2f}.csv'
    )
    
    converters = {
        'palavras_chave': parse_set_safe,
        'patentes_similares': parse_list_safe
    }
    
    dtypes = {
        'lens_id': 'string',
        'year': 'Int64',
        'count_similares': 'Int64',
        'emergente': 'boolean'
    }
    
    print(f"Carregando arquivo: {caminho}...")
    
    try:
        df = pd.read_csv(
            caminho,
            index_col=0,
            converters=converters,
            dtype=dtypes,
            low_memory=False
        )
        
        print("\nArquivo carregado com sucesso!")
        print(f"Total de registros: {len(df):,}")
        
        return df
        
    except FileNotFoundError:
        print(f"ERRO: Arquivo não encontrado")
        return None
    except Exception as e:
        print(f"ERRO: {e}")
        return None


def carregar_analise_tanimoto_ou_cosseno(numero_topicos, numero_palavras_por_topico, 
                              percentual_similaridade, metodo='tanimoto'):
    """Carrega resultados da análise de similaridade Tanimoto/Cossenos."""
    caminho = (
        f'{ANALYSIS_DIR}analise-de-similaridade_{metodo}_'
        f'n{numero_palavras_por_topico}_s{percentual_similaridade:.2f}.csv'
    )
    
    converters = {
        'vetor_caracteristicas': parse_dict_safe,
        'patentes_similares': parse_list_safe
    }
    
    dtypes = {
        'lens_id': 'string',
        'year': 'Int64',
        'count_similares': 'Int64',
        'emergente': 'boolean'
    }
    
    print(f"Carregando arquivo: {caminho}...")
    
    try:
        df = pd.read_csv(
            caminho,
            index_col=0,
            converters=converters,
            dtype=dtypes,
            low_memory=False
        )
        
        print("\nArquivo carregado com sucesso!")
        print(f"Total de registros: {len(df):,}")
        
        return df
        
    except FileNotFoundError:
        print(f"ERRO: Arquivo não encontrado")
        return None
    except Exception as e:
        print(f"ERRO: {e}")
        return None


#### Funções deAnalise com Classificação do IPCR

In [59]:
def safe_convert_list_to_strings(input_data):
    """Converte dados para lista de strings de forma segura."""
    lista_bruta = []
    
    try:
        if isinstance(input_data, str):
            lista_bruta = ast.literal_eval(input_data)
        elif isinstance(input_data, list):
            lista_bruta = input_data
        
        if isinstance(lista_bruta, list):
            return [str(item).strip() for item in lista_bruta if pd.notna(item)]
        else:
            return []
            
    except (ValueError, SyntaxError, TypeError):
        return []


def analisar_com_ipcr(df_main, filtered_ipcr):
    """Analisa patentes emergentes verificando classificação IPCR."""
    df_lookup = filtered_ipcr[['lens_id', 'new_ipcr']].copy()
    df_lookup['lens_id'] = df_lookup['lens_id'].astype(str).str.strip()
    
    patentes_com_new_ipcr = set(
        df_lookup[df_lookup['new_ipcr'] == True]['lens_id']
    )
    
    df_resultado = df_main.copy()
    df_resultado['emergent_new_ipcr'] = False
    
    idx_emergente = df_resultado[df_resultado['emergente']].index
    
    s_similares_lista = df_resultado.loc[idx_emergente, 'patentes_similares'].apply(
        safe_convert_list_to_strings
    )
    
    s_explodida = s_similares_lista.explode().dropna()
    s_matches = s_explodida.isin(patentes_com_new_ipcr)
    
    df_comparacoes = pd.DataFrame({
        'patente_similar_verificada': s_explodida,
        'teve_new_ipcr': s_matches
    })
    df_comparacoes['patente_original_lens_id'] = df_comparacoes.index.map(
        df_resultado['lens_id']
    )
    df_comparacoes = df_comparacoes[[
        'patente_original_lens_id', 
        'patente_similar_verificada', 
        'teve_new_ipcr'
    ]].reset_index(drop=True)
    
    s_resultado_final = s_matches.groupby(level=0).any()
    df_resultado.loc[s_resultado_final.index, 'emergent_new_ipcr'] = s_resultado_final
    
    total_emergent = df_resultado['emergent_new_ipcr'].sum()
    print(f"\nTotal de 'emergent_new_ipcr' = True: {total_emergent}")
    
    return df_resultado, df_comparacoes



#### Funções de treinamento Machine Learning (SVM e XGBOOST)

In [60]:
def treinar_modelos_ml(df_data, df_labels, config):
    """
    Treina e avalia modelos SVM e XGBoost com pré-processamento e
    múltiplas estratégias de balanceamento/otimização.
    
    Args:
        df_data: DataFrame com dados das patentes.
        df_labels: DataFrame com labels (emergent_new_ipcr).
        config: Dicionário com flags de configuração:
            'SMOTE_apply' (bool): Usa SMOTE para oversampling.
            'OPTIMIZE_SVM' (bool): Usa GridSearchCV para otimizar o SVM.
            'USE_GRID_SEARCH' (bool): Usa GridSearchCV para otimizar o XGBoost.
            'USE_GRID_SEARCH_PARAMS' (bool): Usa parâmetros otimizados pré-definidos.
    """
    
    # --- 0. Constante de Aleatoriedade ---
    # Usando 42 para consistência com o teu primeiro script
    RANDOM_STATE = 42 
    
    print(f"--- Dados Iniciais (Patentes): {df_data.shape} ---")
    print(f"--- Rótulos Iniciais (Labels): {df_labels.shape} ---")
    
    # --- 1. Preparação dos Dados ---
    df_merged = pd.merge(df_data, df_labels, on='lens_id')
    print(f"\n--- Dados Combinados: {df_merged.shape} ---")
    
    text_features = ['title_abstract', 'inventor_names']
    numeric_features = ['year', 'patent_count', 'family_count', 'claims_count']
    categorical_features = [
        'jurisdiction', 'kind', 'lang', 'publication_type', 'patent_status'
    ]
    
    df_merged[text_features] = df_merged[text_features].fillna('')
    df_merged[numeric_features] = df_merged[numeric_features].fillna(0)
    df_merged[categorical_features] = df_merged[categorical_features].fillna('Missing')
    
    X = df_merged[text_features + numeric_features + categorical_features]
    y_raw = df_merged['emergent_new_ipcr']
    
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    
    # --- 2. Engenharia de Features (Pré-processamento) ---
    text_transformer = TfidfVectorizer(
        max_features=5000, 
        ngram_range=(1, 2)
    )
    numeric_transformer = StandardScaler()
    categorical_transformer = OneHotEncoder(handle_unknown='ignore')
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('text', text_transformer, text_features[0]),
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ]
    )
    
    # --- 3. Divisão dos Dados (Treino e Teste) ---
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    
    count_neg = np.sum(y_train == 0)
    count_pos = np.sum(y_train == 1)
    # Evitar divisão por zero se não houver positivos (embora raro)
    scale_weight = count_neg / count_pos if count_pos > 0 else 1 
    
    print(f"\nDivisão dos dados de TREINO:")
    print(f"  Classe Negativa (False): {count_neg}")
    print(f"  Classe Positiva (True):  {count_pos}")
    print(f"  Proporção (scale_pos_weight): {scale_weight:.2f}")

    # Variáveis para guardar resultados e modelos
    svm_pipeline = None
    y_pred_svm = None
    svm_best_params = None
    
    xgb_pipeline = None
    y_pred_xgb = None
    xgb_best_params = None

    # --- 4. Modelo 1: Treinamento SVM ---
    
    if config.get('SMOTE_apply', False):
        print("\n--- Treinando SVM (com SMOTE) ---")
        svm_pipeline = ImbPipeline(steps=[
            ('preprocessor', preprocessor),
            ('smote', SMOTE(random_state=RANDOM_STATE)),
            ('classifier', SVC(kernel='linear', random_state=RANDOM_STATE))
        ])
        svm_pipeline.fit(X_train, y_train)
        y_pred_svm = svm_pipeline.predict(X_test)

    elif config.get('OPTIMIZE_SVM', False):
        print("\n--- Treinando SVM (Otimizado com GridSearchCV) ---")
        svm_pipeline_for_grid = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', SVC(random_state=RANDOM_STATE, class_weight='balanced'))
        ])
        
        svm_param_grid = {
            'classifier__kernel': ['linear', 'rbf'],
            'classifier__C': [0.1, 1, 10]
        }
        
        svm_grid_search = GridSearchCV(estimator=svm_pipeline_for_grid,
                                       param_grid=svm_param_grid,
                                       cv=3,
                                       scoring='f1_macro',
                                       n_jobs=-1,
                                       verbose=2)
        
        print("Iniciando o GridSearch para o SVM...")
        svm_grid_search.fit(X_train, y_train)
        
        svm_best_params = svm_grid_search.best_params_
        svm_pipeline = svm_grid_search.best_estimator_ # Salva o melhor modelo
        
        print("Avaliando o melhor modelo SVM...")
        y_pred_svm = svm_pipeline.predict(X_test)
        
    elif config.get('USE_GRID_SEARCH_PARAMS', False):
        print("\n--- Treinando SVM (Balanceado com Parâmetros Otimizados) ---")
        svm_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', SVC(
                kernel='rbf',
                random_state=RANDOM_STATE,
                class_weight='balanced',
                C=1
            ))
        ])
        svm_pipeline.fit(X_train, y_train)
        y_pred_svm = svm_pipeline.predict(X_test)

    else:
        print("\n--- Treinando SVM (Balanceado - Padrão) ---")
        svm_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', SVC(
                kernel='linear',
                random_state=RANDOM_STATE,
                class_weight='balanced'
            ))
        ])
        svm_pipeline.fit(X_train, y_train)
        y_pred_svm = svm_pipeline.predict(X_test)

    
    # --- 5. Modelo 2: Treinamento XGBoost ---
    
    if config.get('SMOTE_apply', False):
        print("\n--- Treinando XGBoost (com SMOTE) ---")
        xgb_pipeline = ImbPipeline(steps=[
            ('preprocessor', preprocessor),
            ('smote', SMOTE(random_state=RANDOM_STATE)),
            ('classifier', XGBClassifier(
                use_label_encoder=False,
                eval_metric='logloss',
                random_state=RANDOM_STATE,
                n_estimators=100
            ))
        ])
        xgb_pipeline.fit(X_train, y_train)
        y_pred_xgb = xgb_pipeline.predict(X_test)

    elif config.get('USE_GRID_SEARCH', False):
        print("\n--- Treinando XGBoost (Otimizado com GridSearchCV) ---")
        xgb_pipeline_for_grid = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', XGBClassifier(
                use_label_encoder=False,
                eval_metric='logloss',
                random_state=RANDOM_STATE,
                scale_pos_weight=scale_weight
            ))
        ])
        
        param_grid = {
            'classifier__n_estimators': [100, 250, 500],
            'classifier__max_depth': [3, 5, 7],
            'classifier__learning_rate': [0.1, 0.05]
        }
        
        grid_search = GridSearchCV(estimator=xgb_pipeline_for_grid,
                                   param_grid=param_grid,
                                   cv=3,
                                   scoring='f1_macro',
                                   n_jobs=-1,
                                   verbose=2)
        
        print("Iniciando o GridSearch para o XGBoost...")
        grid_search.fit(X_train, y_train)
        
        xgb_best_params = grid_search.best_params_
        xgb_pipeline = grid_search.best_estimator_ # Salva o melhor modelo
        
        print("Avaliando o melhor modelo XGBoost...")
        y_pred_xgb = xgb_pipeline.predict(X_test)
        
    elif config.get('USE_GRID_SEARCH_PARAMS', False):
        print("\n--- Treinando XGBoost (Balanceado com Parâmetros Otimizados) ---")
        xgb_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', XGBClassifier(
                use_label_encoder=False,
                eval_metric='logloss',
                random_state=RANDOM_STATE,
                n_estimators=500,
                scale_pos_weight=scale_weight,
                max_depth=3,
                learning_rate=0.1
            ))
        ])
        xgb_pipeline.fit(X_train, y_train)
        y_pred_xgb = xgb_pipeline.predict(X_test)

    else:
        print("\n--- Treinando XGBoost (Balanceado - Padrão) ---")
        xgb_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', XGBClassifier(
                use_label_encoder=False,
                eval_metric='logloss',
                random_state=RANDOM_STATE,
                n_estimators=100,
                scale_pos_weight=scale_weight
            ))
        ])
        xgb_pipeline.fit(X_train, y_train)
        y_pred_xgb = xgb_pipeline.predict(X_test)
    
    
    # --- 6. Avaliação ---
    target_names = le.classes_.astype(str)
    
    print("\n\n" + "="*80)
    print("RESULTADOS FINAIS DA AVALIAÇÃO")
    print("="*80)
    
    print("\n--- Modelo SVM ---")
    if config.get('OPTIMIZE_SVM', False) and svm_best_params:
         print(f"Melhores Parâmetros Encontrados: {svm_best_params}")
    print(f"Acurácia: {accuracy_score(y_test, y_pred_svm):.4f}")
    print("Relatório de Classificação:")
    print(classification_report(y_test, y_pred_svm, target_names=target_names))
    
    print("\n--- Modelo XGBoost ---")
    if config.get('USE_GRID_SEARCH', False) and xgb_best_params:
         print(f"Melhores Parâmetros Encontrados: {xgb_best_params}")
    print(f"Acurácia: {accuracy_score(y_test, y_pred_xgb):.4f}")
    print("Relatório de Classificação:")
    print(classification_report(y_test, y_pred_xgb, target_names=target_names))
    
    return svm_pipeline, xgb_pipeline, (X_test, y_test, y_pred_svm, y_pred_xgb)

In [ ]:
def treinar_modelos_ml_alternativo(df_data, df_labels, config):
    """
    Treina modelos SVM e XGBoost para classificação.
    
    Args:
        df_data: DataFrame com dados das patentes
        df_labels: DataFrame com labels (emergent_new_ipcr)
        config: Dicionário com configurações dos modelos
    """
    print(f"--- Dados Iniciais (Patentes): {df_data.shape} ---")
    print(f"--- Rótulos Iniciais (Labels): {df_labels.shape} ---")
    
    df_merged = pd.merge(df_data, df_labels, on='lens_id')
    print(f"\n--- Dados Combinados: {df_merged.shape} ---")
    
    text_features = ['title_abstract', 'inventor_names']
    numeric_features = ['year', 'patent_count', 'family_count', 'claims_count']
    categorical_features = [
        'jurisdiction', 'kind', 'lang', 'publication_type', 'patent_status'
    ]
    
    df_merged[text_features] = df_merged[text_features].fillna('')
    df_merged[numeric_features] = df_merged[numeric_features].fillna(0)
    df_merged[categorical_features] = df_merged[categorical_features].fillna('Missing')
    
    X = df_merged[text_features + numeric_features + categorical_features]
    y_raw = df_merged['emergent_new_ipcr']
    
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    
    # --- Engenharia de Features (Pré-processamento) ---
    # (Usando a nossa melhor versão com ngram_range=(1, 2))
    text_transformer = TfidfVectorizer(
        max_features=5000, 
        ngram_range=(1, 2)
    )
    numeric_transformer = StandardScaler()
    categorical_transformer = OneHotEncoder(handle_unknown='ignore')
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('text', text_transformer, text_features[0]),
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ]
    )
    
    # --- Divisão dos Dados (Treino e Teste) ---
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    
    count_neg = np.sum(y_train == 0)
    count_pos = np.sum(y_train == 1)
    scale_weight = count_neg / count_pos
    
    print(f"\nDivisão dos dados de TREINO:")
    print(f"  Classe Negativa (False): {count_neg}")
    print(f"  Classe Positiva (True):  {count_pos}")
    print(f"  Proporção (scale_pos_weight): {scale_weight:.2f}")
    
    # Treinamento SVM
    if config.get('SMOTE_apply', False):
        print("\n--- Treinando SVM (com SMOTE) ---")
        svm_pipeline = ImbPipeline(steps=[
            ('preprocessor', preprocessor),
            ('smote', SMOTE(random_state=RANDOM_STATE)),
            ('classifier', SVC(kernel='linear', random_state=RANDOM_STATE))
        ])
    else:
        if config.get('USE_GRID_SEARCH_PARAMS', True):
            print("\n--- Treinando SVM (Balanceado com Parâmetros Otimizados) ---")
            svm_pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('classifier', SVC(
                    kernel='rbf',
                    random_state=RANDOM_STATE,
                    class_weight='balanced',
                    C=1
                ))
            ])
        else:
            print("\n--- Treinando SVM (Balanceado) ---")
            svm_pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('classifier', SVC(
                    kernel='linear',
                    random_state=RANDOM_STATE,
                    class_weight='balanced'
                ))
            ])
    
    svm_pipeline.fit(X_train, y_train)
    y_pred_svm = svm_pipeline.predict(X_test)
    
    # Treinamento XGBoost
    if config.get('SMOTE_apply', False):
        print("\n--- Treinando XGBoost (com SMOTE) ---")
        xgb_pipeline = ImbPipeline(steps=[
            ('preprocessor', preprocessor),
            ('smote', SMOTE(random_state=RANDOM_STATE)),
            ('classifier', XGBClassifier(
                use_label_encoder=False,
                eval_metric='logloss',
                random_state=RANDOM_STATE,
                n_estimators=100
            ))
        ])
    else:
        if config.get('USE_GRID_SEARCH_PARAMS', True):
            print("\n--- Treinando XGBoost (Balanceado com Parâmetros Otimizados) ---")
            xgb_pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('classifier', XGBClassifier(
                    use_label_encoder=False,
                    eval_metric='logloss',
                    random_state=RANDOM_STATE,
                    n_estimators=500,
                    scale_pos_weight=scale_weight,
                    max_depth=3,
                    learning_rate=0.1
                ))
            ])
        else:
            print("\n--- Treinando XGBoost (Balanceado) ---")
            xgb_pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('classifier', XGBClassifier(
                    use_label_encoder=False,
                    eval_metric='logloss',
                    random_state=RANDOM_STATE,
                    n_estimators=100,
                    scale_pos_weight=scale_weight
                ))
            ])
    
    xgb_pipeline.fit(X_train, y_train)
    y_pred_xgb = xgb_pipeline.predict(X_test)
    
    # Avaliação
    target_names = le.classes_.astype(str)
    
    print("\n\n--- RESULTADOS DA AVALIAÇÃO ---")
    print("\n--- Modelo SVM ---")
    print(f"Acurácia: {accuracy_score(y_test, y_pred_svm):.4f}")
    print("Relatório de Classificação:")
    print(classification_report(y_test, y_pred_svm, target_names=target_names))
    
    print("\n--- Modelo XGBoost ---")
    print(f"Acurácia: {accuracy_score(y_test, y_pred_xgb):.4f}")
    print("Relatório de Classificação:")
    print(classification_report(y_test, y_pred_xgb, target_names=target_names))
    
    return svm_pipeline, xgb_pipeline, (X_test, y_test, y_pred_svm, y_pred_xgb)

## Execução do código

### Configurações de Execução

In [ ]:
TREINAR_MODELOS = False # Treino LDA
PROCESSAR_PATENTES = True
EXECUTAR_ANALISE_JACCARD = True
EXECUTAR_ANALISE_TANIMOTO = True
EXECUTAR_ANALISE_COSSENO = True

# um destes deve ser True
USE_JACCARD_FOR_ML = False
USE_TANIMOTO_FOR_ML = False
USE_COSSENO_FOR_ML = True

EXECUTAR_ML = True # Treino ML

### Carregar Dados

In [63]:
print("="*80)
print("CARREGANDO DADOS")
print("="*80)

df_novo = load_dataset(DATASET_PATH)
print(f"Dataset carregado: {len(df_novo)} patentes")

# Adiciona coluna 'year' se não existir
if 'year' not in df_novo.columns:
    df_novo['year'] = df_novo['date_published'].dt.year

# Remove coluna 'index' se existir
if 'index' in df_novo.columns:
    df_novo = df_novo.drop(columns=['index'])

df_novo = df_novo.reset_index(drop=True)
print(f"Colunas: {df_novo.columns.tolist()}")

CARREGANDO DADOS
Dataset carregado: 25240 patentes
Colunas: ['lens_id', 'jurisdiction', 'doc_number', 'kind', 'date_published', 'doc_key', 'docdb_id', 'lang', 'biblio', 'families', 'legal_status', 'publication_type', 'abstract', 'abstract_text', 'ipcr_triples', 'cpc_triples', 'all_triples', 'inventors_detailed', 'inventor_names', 'invention_title_text', 'patent_count', 'patent_lens_ids', 'patent_status', 'application_expiry_date', 'picked', 'title_abstract', 'tokens', 'year']


### Treinamento LDA

In [64]:
if TREINAR_MODELOS:
    print("\n" + "="*80)
    print("TREINAMENTO DE MODELOS LDA")
    print("="*80)
    
    train_and_save_lda_per_year(
        df=df_novo,
        num_topics=NUMERO_DE_TOPICOS,
        passes=10,
        output_dir=MODELS_DIR
    )
    
    # Exibe os tópicos dos modelos treinados
    print("\n--- Visualizando Tópicos dos Modelos ---")
    display_topics_from_models_with_probs(MODELS_DIR, num_words=WORDS_PER_TOPIC_FOR_DISPLAY)
else:
    print("\n" + "="*80)
    print("TREINAMENTO PULADO (modelos já existem)")
    display_topics_from_models_with_probs(MODELS_DIR, num_words=WORDS_PER_TOPIC_FOR_DISPLAY)
    print("="*80)


TREINAMENTO PULADO (modelos já existem)
Encontrados 26 modelos.

--- Tópicos para o Ano: 1999 ---
Tópico 0: 'porta', (3.82%), 'dobradiça', (1.51%), 'motor', (1.09%), 'elevador', (1.01%), 'carro', (0.72%), 'acionamento', (0.66%), 'pino', (0.64%), 'abertura', (0.63%)
Tópico 1: 'invenção', (0.93%), 'elemento', (0.85%), 'material', (0.82%), 'fixação', (0.77%), 'conjunto', (0.73%), 'patente', (0.63%), 'fixador', (0.61%), 'painel', (0.57%)
Tópico 2: 'inferior', (1.07%), 'água', (0.99%), 'superfície', (0.76%), 'parede', (0.74%), 'extremidade', (0.67%), 'lateral', (0.63%), 'aba', (0.58%), 'possuir', (0.57%)
Tópico 3: 'elemento', (0.78%), 'invenção', (0.77%), 'dispositivo', (0.67%), 'parede', (0.66%), 'painel', (0.60%), 'tubo', (0.60%), 'fecho', (0.59%), 'l', (0.56%)
Tópico 4: 'construção', (1.23%), 'elemento', (1.07%), 'parede', (1.03%), 'invenção', (0.96%), 'estrutura', (0.69%), 'vedação', (0.58%), 'patente', (0.57%), 'material', (0.56%)

--- Tópicos para o Ano: 2000 ---
Tópico 0: 'porta', (

### Processamento das Patentes

In [65]:
if PROCESSAR_PATENTES:
    print("\n" + "="*80)
    print("PROCESSAMENTO DE PATENTES")
    print("="*80)
    
    analisar_e_salvar_por_ano_aprimorado(
        df=df_novo,
        modelos_dir=MODELS_DIR,
        num_topics=NUMERO_DE_TOPICOS,
        output_dir=RESULTS_DIR,
        ignorar_existentes=True
    )
else:
    print("\n" + "="*80)
    print("PROCESSAMENTO PULADO (resultados já existem)")
    print("="*80)


PROCESSAMENTO DE PATENTES
Iniciando análise para os anos: [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

--- Ano 1999 já processado. Pulando... ---

--- Ano 2000 já processado. Pulando... ---

--- Ano 2001 já processado. Pulando... ---

--- Ano 2002 já processado. Pulando... ---

--- Ano 2003 já processado. Pulando... ---

--- Ano 2004 já processado. Pulando... ---

--- Ano 2005 já processado. Pulando... ---

--- Ano 2006 já processado. Pulando... ---

--- Ano 2007 já processado. Pulando... ---

--- Ano 2008 já processado. Pulando... ---

--- Ano 2009 já processado. Pulando... ---

--- Ano 2010 já processado. Pulando... ---

--- Ano 2011 já processado. Pulando... ---

--- Ano 2012 já processado. Pulando... ---

--- Ano 2013 já processado. Pulando... ---

--- Ano 2014 já processado. Pulando... ---

--- Ano 2015 já processado. Pulando... ---

--- Ano 2016 já processado. Pulando

### Dataframe com os Resultados (df_completo)

In [66]:
print("\n" + "="*80)
print("PASSO 4: CONCATENANDO RESULTADOS")
print("="*80)

df_completo = concatenar_resultados_por_ano(
    diretorio_resultados=RESULTS_DIR,
    formato='auto'
)

if not df_completo.empty:
    # Converte colunas JSON para listas
    df_completo = converter_json_para_listas(df_completo)
    print("\nDados concatenados e convertidos com sucesso!")
    print(f"Total de patentes: {len(df_completo)}")
else:
    print("\nErro: Nenhum dado foi concatenado!")
    raise ValueError("Falha ao concatenar resultados")


PASSO 4: CONCATENANDO RESULTADOS
Encontrados 26 arquivos para concatenar
✓ resultados_1999.parquet (Parquet) - 223 registros
✓ resultados_2000.parquet (Parquet) - 1166 registros
✓ resultados_2001.parquet (Parquet) - 1084 registros
✓ resultados_2002.parquet (Parquet) - 899 registros
✓ resultados_2003.parquet (Parquet) - 764 registros
✓ resultados_2004.parquet (Parquet) - 958 registros
✓ resultados_2005.parquet (Parquet) - 907 registros
✓ resultados_2006.parquet (Parquet) - 800 registros
✓ resultados_2007.parquet (Parquet) - 654 registros
✓ resultados_2008.parquet (Parquet) - 768 registros
✓ resultados_2009.parquet (Parquet) - 603 registros
✓ resultados_2010.parquet (Parquet) - 652 registros
✓ resultados_2011.parquet (Parquet) - 805 registros
✓ resultados_2012.parquet (Parquet) - 394 registros
✓ resultados_2013.parquet (Parquet) - 826 registros
✓ resultados_2014.parquet (Parquet) - 526 registros
✓ resultados_2015.parquet (Parquet) - 901 registros
✓ resultados_2016.parquet (Parquet) - 12

### Analise de Patentes Jaccard

In [67]:
if EXECUTAR_ANALISE_JACCARD:
    print("\n" + "="*80)
    print("ANÁLISE DE SIMILARIDADE (JACCARD)")
    print("="*80)
    
    df_analise_jaccard = analise_similaridade_jaccard(
        df_completo=df_completo,
        numero_palavras_por_topico=NUM_PALAVRAS_POR_TOPICO,
        percentual_similaridade=PERCENTUAL_SIMILARIDADE,
        numero_topicos=NUMERO_DE_TOPICOS
    )
    print("Análise Jaccard concluída")
else:
    print("\n" + "="*80)
    print("CARREGANDO ANÁLISE JACCARD EXISTENTE")
    print("="*80)
    
    df_analise_jaccard = carregar_analise_jaccard(
        numero_topicos=NUMERO_DE_TOPICOS,
        numero_palavras_por_topico=NUM_PALAVRAS_POR_TOPICO,
        percentual_similaridade=PERCENTUAL_SIMILARIDADE
    )
    if df_analise_jaccard is None:
        print("Erro ao carregar análise Jaccard!")
    else:
        print("Análise Jaccard carregada")


ANÁLISE DE SIMILARIDADE (JACCARD)
Iniciando extração vetorizada de palavras-chave...
Limpando palavras-chave...
Assinaturas de palavras criadas com sucesso.

Iniciando comparação de similaridade otimizada...
Comparando ano 1999 com 2000...
Comparando ano 2000 com 2001...
Comparando ano 2001 com 2002...
Comparando ano 2002 com 2003...
Comparando ano 2003 com 2004...
Comparando ano 2004 com 2005...
Comparando ano 2005 com 2006...
Comparando ano 2006 com 2007...
Comparando ano 2007 com 2008...
Comparando ano 2008 com 2009...
Comparando ano 2009 com 2010...
Comparando ano 2010 com 2011...
Comparando ano 2011 com 2012...
Comparando ano 2012 com 2013...
Comparando ano 2013 com 2014...
Comparando ano 2014 com 2015...
Comparando ano 2015 com 2016...
Comparando ano 2016 com 2017...
Comparando ano 2017 com 2018...
Comparando ano 2018 com 2019...
Comparando ano 2019 com 2020...
Comparando ano 2020 com 2021...
Comparando ano 2021 com 2022...
Comparando ano 2022 com 2023...
Comparando ano 2023 com

### Analise de Patentes Tanimoto

In [68]:
if EXECUTAR_ANALISE_TANIMOTO:
        print("\n" + "="*80)
        print(f"ANÁLISE DE SIMILARIDADE (TANIMOTO)")
        print("="*80)
        
        df_analise_tanimoto = analise_similaridade_tanimoto_or_cosseno(
            df_completo=df_completo,
            numero_palavras_por_topico=NUM_PALAVRAS_POR_TOPICO,
            percentual_similaridade=PERCENTUAL_SIMILARIDADE,
            numero_topicos=NUMERO_DE_TOPICOS,
            metodo='tanimoto'
        )
        print(f"Análise tanimoto concluída")
else:
        print("\n" + "="*80)
        print(f"CARREGANDO ANÁLISE TANIMOTO EXISTENTE")
        print("="*80)
        
        df_analise_tanimoto = carregar_analise_tanimoto_ou_cosseno(
            numero_topicos=NUMERO_DE_TOPICOS,
            numero_palavras_por_topico=NUM_PALAVRAS_POR_TOPICO,
            percentual_similaridade=PERCENTUAL_SIMILARIDADE,
            metodo='tanimoto'
        )
        if df_analise_tanimoto is None:
            print(f"Erro ao carregar análise tanimoto!")
        else:
            print(f"Análise tanimoto carregada")


ANÁLISE DE SIMILARIDADE (TANIMOTO)
Iniciando extração vetorizada de vetores de características...


C:\Users\pedro\AppData\Local\Temp\ipykernel_4816\3152574554.py:224: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  vetores_series = df_final_palavras.groupby('lens_id').apply(


Vetores criados com sucesso.
Pré-calculando magnitudes para o método 'tanimoto'...
Magnitudes calculadas.

Iniciando comparação de similaridade (tanimoto) otimizada...
Comparando ano 1999 com 2000...
Comparando ano 2000 com 2001...
Comparando ano 2001 com 2002...
Comparando ano 2002 com 2003...
Comparando ano 2003 com 2004...
Comparando ano 2004 com 2005...
Comparando ano 2005 com 2006...
Comparando ano 2006 com 2007...
Comparando ano 2007 com 2008...
Comparando ano 2008 com 2009...
Comparando ano 2009 com 2010...
Comparando ano 2010 com 2011...
Comparando ano 2011 com 2012...
Comparando ano 2012 com 2013...
Comparando ano 2013 com 2014...
Comparando ano 2014 com 2015...
Comparando ano 2015 com 2016...
Comparando ano 2016 com 2017...
Comparando ano 2017 com 2018...
Comparando ano 2018 com 2019...
Comparando ano 2019 com 2020...
Comparando ano 2020 com 2021...
Comparando ano 2021 com 2022...
Comparando ano 2022 com 2023...
Comparando ano 2023 com 2024...
Comparação finalizada.

Resultad

### Análise de Patentes Cossenos

In [69]:
if EXECUTAR_ANALISE_COSSENO:
    print("\n" + "="*80)
    print(f"ANÁLISE DE SIMILARIDADE (COSSENOS)")
    print("="*80)
    
    df_analise_cosseno = analise_similaridade_tanimoto_or_cosseno(
        df_completo=df_completo,
        numero_palavras_por_topico=NUM_PALAVRAS_POR_TOPICO,
        percentual_similaridade=PERCENTUAL_SIMILARIDADE,
        numero_topicos=NUMERO_DE_TOPICOS,
        metodo='cossenos'
    )
    print(f"Análise cossenos concluída")
else:
    print("\n" + "="*80)
    print(f"CARREGANDO ANÁLISE COSSENO EXISTENTE")
    print("="*80)
    
    df_analise_cosseno = carregar_analise_tanimoto_ou_cosseno(
        numero_topicos=NUMERO_DE_TOPICOS,
        numero_palavras_por_topico=NUM_PALAVRAS_POR_TOPICO,
        percentual_similaridade=PERCENTUAL_SIMILARIDADE,
        metodo='cossenos'
    )
    if df_analise_cosseno is None:
        print(f"Erro ao carregar análise cossenos!")
    else:
        print(f"Análise cossenos carregada")


ANÁLISE DE SIMILARIDADE (COSSENOS)
Iniciando extração vetorizada de vetores de características...


C:\Users\pedro\AppData\Local\Temp\ipykernel_4816\3152574554.py:224: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  vetores_series = df_final_palavras.groupby('lens_id').apply(


Vetores criados com sucesso.
Pré-calculando magnitudes para o método 'cossenos'...
Magnitudes calculadas.

Iniciando comparação de similaridade (cossenos) otimizada...
Comparando ano 1999 com 2000...
Comparando ano 2000 com 2001...
Comparando ano 2001 com 2002...
Comparando ano 2002 com 2003...
Comparando ano 2003 com 2004...
Comparando ano 2004 com 2005...
Comparando ano 2005 com 2006...
Comparando ano 2006 com 2007...
Comparando ano 2007 com 2008...
Comparando ano 2008 com 2009...
Comparando ano 2009 com 2010...
Comparando ano 2010 com 2011...
Comparando ano 2011 com 2012...
Comparando ano 2012 com 2013...
Comparando ano 2013 com 2014...
Comparando ano 2014 com 2015...
Comparando ano 2015 com 2016...
Comparando ano 2016 com 2017...
Comparando ano 2017 com 2018...
Comparando ano 2018 com 2019...
Comparando ano 2019 com 2020...
Comparando ano 2020 com 2021...
Comparando ano 2021 com 2022...
Comparando ano 2022 com 2023...
Comparando ano 2023 com 2024...
Comparação finalizada.

Resultad

### Análise junto da classificação do IPCR

In [70]:
print("\n" + "="*80)
print("ANÁLISE COM CLASSIFICAÇÃO IPCR")
print("="*80)

# Carrega dados IPCR
filtered_ipcr = pd.read_csv(FILTERED_IPCR_PATH, compression='zip')
print(f"Dataset IPCR carregado: {len(filtered_ipcr)} registros")

if df_analise_jaccard is not None:
    # Análise com IPCR - Jaccard
    print("\n--- Analisando patentes emergentes (Jaccard) ---")
    df_resultado_jaccard, df_comparacoes_jaccard = analisar_com_ipcr(
        df_main=df_analise_jaccard,
        filtered_ipcr=filtered_ipcr
    )
    print("Análise IPCR (Jaccard) concluída")

if df_analise_tanimoto is not None:
    # Análise com IPCR - Tanimoto
    print("\n--- Analisando patentes emergentes (Tanimoto) ---")
    df_resultado_tanimoto, df_comparacoes_tanimoto = analisar_com_ipcr(
        df_main=df_analise_tanimoto,
        filtered_ipcr=filtered_ipcr
    )
    print("Análise IPCR (Tanimoto) concluída")

if df_analise_cosseno is not None:
    # Análise com IPCR - Cossenos
    print("\n--- Analisando patentes emergentes (Cossenos) ---")
    df_resultado_cosseno, df_comparacoes_cosseno = analisar_com_ipcr(
        df_main=df_analise_cosseno,
        filtered_ipcr=filtered_ipcr
    )
    print("Análise IPCR (Cossenos) concluída")


ANÁLISE COM CLASSIFICAÇÃO IPCR
Dataset IPCR carregado: 24752 registros

--- Analisando patentes emergentes (Jaccard) ---

Total de 'emergent_new_ipcr' = True: 2
Análise IPCR (Jaccard) concluída

--- Analisando patentes emergentes (Tanimoto) ---

Total de 'emergent_new_ipcr' = True: 143
Análise IPCR (Tanimoto) concluída

--- Analisando patentes emergentes (Cossenos) ---

Total de 'emergent_new_ipcr' = True: 574
Análise IPCR (Cossenos) concluída


### Preparação para o ML

In [71]:
print("\n" + "="*80)
print("PASSO 7: PREPARANDO DADOS PARA ML")
print("="*80)

# Merge com dados completos para ML
colunas_para_ml = ['lens_id', 'family_count', 'claims_count', 
                'ipcr_publication_years', 'new_ipcr']
colunas_existentes = [col for col in colunas_para_ml if col in filtered_ipcr.columns]

df_train = pd.merge(
    df_novo,
    filtered_ipcr[colunas_existentes],
    on='lens_id',
    how='left'
)
print(f"✓ Dataset de treino preparado: {df_train.shape}")


PASSO 7: PREPARANDO DADOS PARA ML
✓ Dataset de treino preparado: (25240, 32)


### Treinamento ML (SVM e XGBoost)

In [ ]:
if EXECUTAR_ML:
    print("\n" + "="*80)
    print("PASSO 8: TREINAMENTO DE MODELOS ML")
    print("="*80)
    
    # --- Configurações do ML ---
    # Escolhe a tua estratégia:
    
    # Estratégia 1: Usar Parâmetros Otimizados
    config_ml = {
        'SMOTE_apply': False,
        'OPTIMIZE_SVM': False,
        'USE_GRID_SEARCH': False,
        'USE_GRID_SEARCH_PARAMS': True  # Usar os melhores parâmetros fixos
    }
    
    # Estratégia 2: Executar GridSearchCV
    # config_ml = {
    #     'SMOTE_apply': False,
    #     'OPTIMIZE_SVM': True,  # <-- Executar GridSearch para SVM
    #     'USE_GRID_SEARCH': True, # <-- Executar GridSearch para XGBoost
    #     'USE_GRID_SEARCH_PARAMS': False
    # }

    # Estratégia 3: Usar SMOTE
    # config_ml = {
    #     'SMOTE_apply': True,
    #     'OPTIMIZE_SVM': False,
    #     'USE_GRID_SEARCH': False,
    #     'USE_GRID_SEARCH_PARAMS': False
    # }

    # Estratégia 4: Usar Modelos Padrão (Balanceados)
    # config_ml = {
    #     'SMOTE_apply': False,
    #     'OPTIMIZE_SVM': False,
    #     'USE_GRID_SEARCH': False,
    #     'USE_GRID_SEARCH_PARAMS': False
    # }
    
    if USE_JACCARD_FOR_ML:
        df_labels = df_resultado_jaccard[['lens_id', 'emergent_new_ipcr']].copy()

        svm_model_jaccard, xgb_model_jaccard, resultados_jaccard = treinar_modelos_ml(
            df_data=df_train,
            df_labels=df_labels,
            config=config_ml
        )
    if USE_TANIMOTO_FOR_ML:
        df_labels = df_resultado_tanimoto[['lens_id', 'emergent_new_ipcr']].copy()
        
        svm_model_tanimoto, xgb_model_tanimoto, resultados_tanimoto = treinar_modelos_ml(
            df_data=df_train,
            df_labels=df_labels,
            config=config_ml
        )
    if USE_COSSENO_FOR_ML:
        df_labels = df_resultado_cosseno[['lens_id', 'emergent_new_ipcr']].copy()
        
        svm_model_cosseno, xgb_model_cosseno, resultados_cosseno = treinar_modelos_ml(
            df_data=df_train,
            df_labels=df_labels,
            config=config_ml
        )
    
    print("\nTreinamento de modelos ML concluído!")
else:
    print("\n" + "="*80)
    print("PASSO 8: TREINAMENTO ML PULADO")
    print("="*80)


PASSO 8: TREINAMENTO DE MODELOS ML
--- Dados Iniciais (Patentes): (25240, 32) ---
--- Rótulos Iniciais (Labels): (25240, 2) ---

--- Dados Combinados: (25240, 33) ---

Divisão dos dados de TREINO:
  Classe Negativa (False): 20078
  Classe Positiva (True):  114
  Proporção (scale_pos_weight): 176.12

--- Treinando SVM (Balanceado com Parâmetros Otimizados) ---

--- Treinando XGBoost (Balanceado com Parâmetros Otimizados) ---


d:\Universidade\S6\RP2\artigo-rp2\analise_patentes\.venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [15:39:24] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)




RESULTADOS FINAIS DA AVALIAÇÃO

--- Modelo SVM ---
Acurácia: 0.9921
Relatório de Classificação:
              precision    recall  f1-score   support

       False       0.99      1.00      1.00      5019
        True       0.13      0.07      0.09        29

    accuracy                           0.99      5048
   macro avg       0.56      0.53      0.54      5048
weighted avg       0.99      0.99      0.99      5048


--- Modelo XGBoost ---
Acurácia: 0.9893
Relatório de Classificação:
              precision    recall  f1-score   support

       False       0.99      1.00      0.99      5019
        True       0.00      0.00      0.00        29

    accuracy                           0.99      5048
   macro avg       0.50      0.50      0.50      5048
weighted avg       0.99      0.99      0.99      5048


✓ Treinamento de modelos ML concluído!


### Resumo

In [73]:
print("\n" + "="*80)
print("RESUMO DA EXECUÇÃO")
print("="*80)
print(f" - Dados carregados: {len(df_novo):,} patentes")
print(f" - Modelos LDA: {NUMERO_DE_TOPICOS} tópicos")
print(f" - Análise de similaridade: {NUM_PALAVRAS_POR_TOPICO} palavras, {PERCENTUAL_SIMILARIDADE:.0%} threshold")

if not df_completo.empty:
    print(f" - Resultados concatenados: {len(df_completo):,} patentes")
    
if 'df_analise_jaccard' in locals():
    emergentes_j = df_analise_jaccard['emergente'].sum()
    print(f" - Patentes emergentes (Jaccard): {emergentes_j:,}")
    
if 'df_analise_tanimoto' in locals():
    emergentes_t = df_analise_tanimoto['emergente'].sum()
    print(f" - Patentes emergentes (tanimoto): {emergentes_t:,}")
if 'df_analise_cosseno' in locals():
    emergentes_c = df_analise_cosseno['emergente'].sum()
    print(f" - Patentes emergentes (cosseno): {emergentes_c:,}")
if 'df_resultado_jaccard' in locals():
    ipcr_j = df_resultado_jaccard['emergent_new_ipcr'].sum()
    print(f" - Emergentes com novo IPCR (Jaccard): {ipcr_j:,}")
if 'df_resultado_tanimoto' in locals():
    ipcr_t = df_resultado_tanimoto['emergent_new_ipcr'].sum()
    print(f" - Emergentes com novo IPCR (Tanimoto): {ipcr_t:,}")
if 'df_resultado_cosseno' in locals():
    ipcr_c = df_resultado_cosseno['emergent_new_ipcr'].sum()
    print(f" - Emergentes com novo IPCR (Cosseno): {ipcr_c:,}")
print("="*80)
print("EXECUÇÃO CONCLUÍDA COM SUCESSO!")
print("="*80)


RESUMO DA EXECUÇÃO
 - Dados carregados: 25,240 patentes
 - Modelos LDA: 5 tópicos
 - Análise de similaridade: 5 palavras, 95% threshold
 - Resultados concatenados: 25,240 patentes
 - Patentes emergentes (Jaccard): 25
 - Patentes emergentes (tanimoto): 980
 - Patentes emergentes (cosseno): 3,074
 - Emergentes com novo IPCR (Jaccard): 2
 - Emergentes com novo IPCR (Tanimoto): 143
 - Emergentes com novo IPCR (Cosseno): 574
EXECUÇÃO CONCLUÍDA COM SUCESSO!
